![Dwengo](images/dwengo.png)

# Recognizing emotions
In this activity, you will develop an AI system that is capable of recognizing emotions. This way, you will learn, step by step, various principles of AI and machine learning.

## Prerequisites
To get started with this notebook, you need basic knowledge of programming in Python. In this notebook, you'll use data types, operators, structures, and functions. If you're not sure you have sufficient knowledge of Python for this notebook, you can visit [dwengo.org/python](https://dwengo.org/python). There, the basic principles are explained step by step.

## Installing and importing the necessary modules
Before you start building the system, you first load some modules. These contain preprogrammed functions that you will need later.

In [ ]:
# modules installeren
!pip install opencv-contrib-python==4.10.0.84

In [ ]:
# modules inladen
import matplotlib.pyplot as plt
from PIL import Image
from scripts import helpers
import numpy as np
from sklearn.model_selection import train_test_split

import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, InputLayer, BatchNormalization

## Data collection
AI systems learn rules from data. The quality of an AI system therefore also depends on the quality of the dataset. There are a number of conditions your dataset must meet to be considered high quality.
* **Correct labels**: The information in the dataset must be correct. Images of cats must receive the label 'cat', images of dogs, the label 'dog'. Images that have an incorrect label will confuse the AI system.* **Complete information**: The dataset must contain all the information required to solve the problem. If you want to detect cats and dogs, for example, it's best to have photos of all cat and dog breeds.* **Unique elements**: The elements in the dataset must be unique. Therefore, each photo of a cat or a dog appears only once in the dataset. Photos that appear multiple times do not help improve the system.* **Balanced**: There are an equal number of examples for each type of element. For example, as many examples of cats as of dogs.* **Ethical**: Did you obtain the dataset in an ethical manner? Is the data subject to copyright? Does the data contain personal data?
In this activity, you will compile a dataset yourself. This way, you can safeguard the quality. You will notice that assembling a dataset is a lot of work. Putting together a high-quality dataset is often one of the main challenges in developing an AI system.

### Drawing emotions
You will build a system that can detect emotions. You won't do this directly on photos of people; you would need too complex a system for that. You start by detecting the emotions of smileys: you try to distinguish **smiling and surprised smileys** from each other.<br> Here you see an example of a smiling and a surprised smiley.
![](images/voorbeeld_blij.png)

![](images/voorbeeld_verbaasd.png)
To train the AI system, you need about a hundred happy and about a hundred surprised smileys. Yes, you will have to draw them yourself, and that is quite a lot of work. Don’t worry, there is a method provided that lets you do this easily using a template. The template contains a grid. In each cell of the grid you draw a smiley. **Per sheet, you draw only one emotion**, so either all happy smileys or all surprised ones.<br>Here you can see an example of such a filled-in template.
![](images/voorbeeld_raster_blij.jpg)

**Assignment**: Print the document *raster.pdf* 10 times in A3 format: 5 sheets for happy smileys and 5 sheets for surprised smileys. Fill in the grid by drawing smileys. Tip: you can distribute the sheets among different people, so you spread the drawing work.

**Assignment**: Take photos of the completed templates. Make sure that the four markers at the corners of the grid are visible in the photo. Also make sure that the photo captures the smileys and the markers in clear focus.

## Loading the data
Now that you have collected a dataset, you need to load it into Python.On the left in the file explorer you see a folder named *dataset*. Inside it are two subfolders named *happy* and *surprised*.
**Assignment**: Add the photos of the grids to the correct folder. **NOTE! The images must have a .jpg, .png or .jpeg extension!**
To upload a file to that folder, click the upload icon at the top of the file explorer. In the image below, that icon is indicated with a green arrow.
![](images/hoe_uploaden.png)

There are a number of functions provided to make it easier to process the data. In the next code cell, a function is called that takes two parameters. The first parameter is the folder with images of happy smileys, the second parameter is the label of the items in that folder.

In [ ]:
# laad alle foto's in map 'dataset/blij' in en geef ze label 'blij'
# rasters met afbeeldingen worden automatisch in stukjes geknipt
afbeeldingen_blij, labels_blij = helpers.laadt_bestanden_in_map_met_label("dataset/blij", label="blij")

Both the images of happy smileys and their label are referenced using a variable.
**Question**: Which variable refers to the images and which to the labels?<br>**Answer**:

Now that you’ve loaded the images of happy smileys, take a look at what they look like. The following code cell contains the code to display various properties of the dataset.

In [ ]:
print(f"De dataset bevat {len(afbeeldingen_blij)} afbeeldingen met label 'blij'")
print(f"De labels zijn: {labels_blij}")
print(f"De eerste afbeelding heeft een grootte van {afbeeldingen_blij[0].shape}")
print("De eerste zes afbeeldingen zien er als volgt uit:")
helpers.toon_afbeeldingen(afbeeldingen_blij, labels_blij, max_afbeeldingen=6)

**Task**: Complete the following code cell to reference both the images of surprised smileys and their label with a variable.

In [ ]:
# vul de code aan op de plaatsen waar ___ staat
# naar afbeeldingen van verbaasde smileys verwijzen met een variabele
afbeeldingen_verbaasd, labels_verbaasd = helpers.laadt_bestanden_in_map_met_label(___, label=___)

**Assignment**: Also complete the following code cell to display the information about the surprised smileys.

In [ ]:
# vul de code aan op de plaatsen waar ___ staat
# informatie over verbaasde smileys laten zien
print(f"De dataset bevat {___} afbeeldingen met label '___'")
print(f"De labels zijn: {___}")
print(f"De eerste afbeelding heeft een grootte van {___}")
print("De eerste zes afbeeldingen zien er als volgt uit:")
helpers.toon_afbeeldingen(___, ___, max_afbeeldingen=6)

## Preparing the data for the AI system
Now that you have loaded the **labeled images** into Python, you process them into a format that the AI system needs. To do so, you go through the following steps.1. Merge happy and surprised images into one dataset.2. Convert the labels from text to numbers.3. Split this dataset into three sets.    * The training set: you use this to train the AI system.    * The validation set: you use this to validate the performance of the AI system during development. The images in this set do not overlap with the training set. This set is needed to see whether the AI system can generalize and thus has not simply memorized the images in the training set.    * The test set: this is used to test the performance of the AI system after development. The images in this set do not overlap with those in the training and validation sets.    

It may not be entirely clear yet why you need these sets. Later in this notebook, this should become clearer. In the following cells, you'll go ahead and start defining the sets.

### Step 1: merging the images and labels
Using the following code, you combine all images into a *NumPy array*. You do the same for all labels.

In [ ]:
afbeeldingen = np.vstack([np.array(afbeeldingen_blij), np.array(afbeeldingen_verbaasd)])
afbeeldingen = np.expand_dims(afbeeldingen, axis=-1)
labels = np.concatenate([np.array(labels_blij), np.array(labels_verbaasd)])

Show information about the arrays.

In [ ]:
print(f"De dataset bevat {afbeeldingen.shape[0]} afbeeldingen")
print(f"Er zijn {len(labels)} labels")
print(f"De eerste afbeelding heeft een grootte van {afbeeldingen[0].shape}")

**Assignment**: Check the number of images in the dataset. Does that match the sum of the number of happy images and the number of surprised images?

### Step 2: Convert the labels from text to numbers.
Because computers can perform calculations with numbers more quickly and efficiently, you convert the labels from text to numbers using **one-hot** encoding. You will represent each label with two digits. The first digit is a 1 when the label is *happy* and a 0 when the label is *surprised*. The second digit is a 0 when the label is *happy* and a 1 when the label is *surprised*.<br>Here is an example:
![](images/formaat_labels.png)

In [ ]:
# labels one-hot encoderen
labels_one_hot = helpers.one_hot_encode_labels(labels, ["blij", "verbaasd"])

Now that the labels are numeric, show 10 random images with their new label.

In [ ]:
# 10 willekeurige indices genereren
random_indices = np.random.randint(0, len(labels), 10)
# deze 10 willekeurige afbeeldingen laten zien
helpers.toon_afbeeldingen(afbeeldingen[random_indices], labels_one_hot[random_indices], max_afbeeldingen=10)

### Step 3: split the dataset into training, test, and validation sets.
To split the dataset into these three sets, use the *train_test_split()* function from the *sklearn* module. You use it twice: first to split off a test set, then to obtain a training and validation set.

#### Obtaining the test set
By running the following code cell, 20% of the dataset will be randomly selected as a test set.

In [ ]:
trainval_afbeeldingen, test_afbeeldingen, trainval_labels, test_labels = train_test_split(afbeeldingen, labels_one_hot, test_size=0.2)

Check the format of the test set and of the collection of other images that will be used to develop the system.

In [ ]:
print(f"De ontwikkelingsdataset bevat {trainval_afbeeldingen.shape[0]} afbeeldingen")
print(f"De testverzameling bevat {test_afbeeldingen.shape[0]} afbeeldingen")

**Task**: Complete the code below so that the `trainval_afbeeldingen` and `trainval_labels` are split into a training set and a validation set. 10% of these images must be used as the validation set.

In [ ]:
# vul de code aan op de plaatsen waar ___ staat
train_afbeeldingen, val_afbeeldingen, train_labels, val_labels = train_test_split(___, ___, test_size=___)

In [ ]:
print(f"De trainingsverzameling bevat {___} afbeeldingen")
print(f"De validatieverzameling bevat {___} afbeeldingen")

## Training the AI system
Now that the dataset is ready, you can train the AI system. To do this, you will use a neural network. In the next cell, a function is defined that specifies the structure of the neural network. The code cell below provides a visual representation of this network.

It is not so important to already know how such an AI system works. You can see the neural network as a system that searches for the rules needed to recognize emotions, or in other words: the system *learns* to recognize the emotions. The network learns these rules by looking at examples. You don't need to understand the details, but know that such a network will transform an image step by step into two numbers. These numbers indicate how confident the system is in its decision. The first number represents how certain the system is that the image contains a happy emotion, the second how certain it is that it contains a surprised emotion.<br>The system will ultimately make a 'prediction' for an image. It will assign an image to the class it is most confident about.

In [ ]:
# structuur vastleggen van het AI-model dat getraind zal worden
def maak_neuraal_netwerk(hoogte_afbeelding, breedte_afbeelding):
    model = Sequential()
    
    model.add(InputLayer(input_shape=(hoogte_afbeelding, breedte_afbeelding, 1)))
    
    model.add(Conv2D(1, (3, 3), activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.1))
    
    model.add(Conv2D(2, (3, 3), activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.1))
    
    model.add(Conv2D(4, (3, 3), activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.1))
    
    model.add(Conv2D(8, (3, 3), activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.1))
    
    model.add(Conv2D(16, (3, 3), activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.1))
    
    model.add(Flatten())
    model.add(Dense(16, activation="relu"))   
    model.add(Dropout(0.1)) 
    model.add(Dense(2, activation="softmax"))
    
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    
    return model

![](images/neural_network_visualized.png)

The instruction in the following code cell calls the function that creates the model.

In [ ]:
model = maak_neuraal_netwerk(afbeeldingen.shape[1], afbeeldingen.shape[2])

Using the `summary()` function, you can display the details of the model.<br>Note that you also see how many *parameters* the system has to learn. The more parameters, the more complex the model is, and thus the more data is needed to train it.

In [ ]:
model.summary()

Now you will train the model using the dataset. When you run the next cell, you will see that the network starts learning from the training set. You will receive various information.* You can see which *epoch* is in progress. This indicates how many times the entire training set has been presented to the network as an example.* The *accuracy* is calculated by dividing the number of correct predictions by the total number of predictions. The higher the accuracy, the better the performance of the network on the training set.* The *loss* indicates how much the network's predictions deviate on average from the true value. The higher the loss, the worse the network's performance on the training set.
For each epoch, the *accuracy* and *loss* are also calculated for the validation set, to see how the system performs on data that differ from the training data.* The higher the *val_accuracy*, the better the network's performance on the validation set.* The higher the *val_loss*, the worse the performance of the network on the validation set.
These values are used to guide the training of the network.

In [ ]:
model.fit(train_afbeeldingen, train_labels, epochs=5, batch_size=1, validation_data=(val_afbeeldingen, val_labels))

Using the following code cell, you can examine the *val_accuracy* in more detail.

In [ ]:
# accuracy op de validatieverzameling berekenen
loss, accuracy = model.evaluate(val_afbeeldingen, val_labels)
print(f"Validatie accuracy: {accuracy}")

The value you will obtain here will always be slightly different. This depends, among other things, on the quality of the dataset used here. In our tests, we often achieve an accuracy of about 0.95, or 95%.

The *accuracy* gives you an idea of the system’s performance but does not tell us how well it can recognize each of the groups. To get more insight into this, you can look at the *confusion matrix*. This matrix indicates how many images from each category were predicted correctly. Below you can see an example of a confusion matrix that we obtained for our model.
![](images/voorbeeld_confusion_matrix.png)
Above you can see that 18 happy images were predicted correctly. Two of the happy images were predicted incorrectly, with the label surprised. All surprised images were predicted correctly.

Run the following code to get an idea of the confusion matrix of the model that you trained.

In [ ]:
# print de confusion matrix van de validatieverzameling
predictions = model.predict(val_afbeeldingen)
confusion_matrix = helpers.create_confusion_matrix_for_one_hot_encoded_labels(val_labels, predictions, ["blij", "verbaasd"])
helpers.visualize_confussion_matrix_in_heatmap(confusion_matrix, ["blij", "verbaasd"])

You can also display an image from the validation set along with the prediction.

In [ ]:
mapped_labels_true = ["blij" if np.argmax(label) == 0 else "verbaasd" for label in val_labels]
mapped_labels_predicted = ["blij" if np.argmax(label) == 0 else "verbaasd" for label in predictions]
mapped_labels_combined = [f"Echt: {mapped_labels_true[i]} \n Voorspeld: {mapped_labels_predicted[i]}" for i in range(len(mapped_labels_true))]
helpers.toon_afbeeldingen(val_afbeeldingen, mapped_labels_combined, max_afbeeldingen=len(val_afbeeldingen))

## Testing the AI system

Now that you have trained the AI system on the training and validation sets, you check whether it also works on images it has not seen before. This is necessary to verify that the model has truly learned to recognize the characteristics of the emotions and has not simply memorized the provided examples. <br>To do this, you use the test set. This contains images that were not used to train the AI system. If the system works on these images, then the model has indeed learned the characteristics of the emotion images. In this case you say that the model can *generalize* to other examples.
To measure the performance of the AI system, there are various metrics you can use. A simple metric is the *accuracy*, which you already know for the training and validation set. Accuracy is the ratio of the number of correctly predicted images to the total number of predictions. In the cell below, you compute the *accuracy* on the test set.

In [ ]:
# accuracy op testverzameling
test_loss, test_accuracy = model.evaluate(test_afbeeldingen, test_labels)
print(f"Test accuracy: {test_accuracy}")

# confusion matrix van de testverzameling
predictions = model.predict(test_afbeeldingen)
confusion_matrix = helpers.create_confusion_matrix_for_one_hot_encoded_labels(test_labels, predictions, ["blij", "verbaasd"])
helpers.visualize_confussion_matrix_in_heatmap(confusion_matrix, ["blij", "verbaasd"])

In [ ]:
# afbeeldingen uit de testverzameling met hun voorspelling
mapped_labels_true = ["blij" if np.argmax(label) == 0 else "verbaasd" for label in test_labels]
mapped_labels_predicted = ["blij" if np.argmax(label) == 0 else "verbaasd" for label in predictions]
mapped_labels_combined = [f"Echt: {mapped_labels_true[i]} \n Voorspeld: {mapped_labels_predicted[i]}" for i in range(len(mapped_labels_true))]
helpers.toon_afbeeldingen(test_afbeeldingen, mapped_labels_combined, max_afbeeldingen=len(test_afbeeldingen))

If the model is not working well yet, you can choose to adjust the model's structure or the training parameters.

But normally you should already have a system that works reasonably well. It is therefore probably not necessary here to adjust the parameters of the model. If the model still does not work well, you can choose to modify the model so that it yields a better result on the validation set. To make sure that your adjustments work well on other data, you then also test it on the test set.

## A custom image
You should now have a system that is relatively good at distinguishing smiling and surprised smileys. You can test how it works again with new drawings. Print the template with the boxes again. Draw smileys in the boxes again. Some of the smileys are surprised, others are smiling. After drawing, take another photo of your sheet and upload it to the folder *dataset/eigen*.
Read in the images just as you did before.

In [ ]:
# laad alle afbeeldingen in map 'dataset/eigen' in en geef ze label 'onbekend'
# rasters op deze afbeeldingen worden automatisch in stukjes geknipt
eigen_afbeeldingen, _ = helpers.laadt_bestanden_in_map_met_label(___, label=___)
eigen_afbeeldingen = np.expand_dims(eigen_afbeeldingen, axis=-1)

Make a prediction on the new image.

In [ ]:
predictions = model.predict(___)

Show the result of the prediction.

In [ ]:
mapped_labels_predicted = ["blij" if np.argmax(label) == 0 else "verbaasd" for label in predictions]
helpers.toon_afbeeldingen(eigen_afbeeldingen, mapped_labels_predicted, max_afbeeldingen=len(eigen_afbeeldingen))

# Challenge
You now have a system that can distinguish between two emotions. You can easily extend this system to three emotions. To do that, you first need to collect additional data. Once you have this data, you can use the code above as a basis to build a new system that can distinguish between three emotions.

# Extension: more than emotions
You can, in principle, use this system to tell any two objects apart. So instead of images of smileys, you could make drawings of cats and dogs. You can then use the same system to distinguish these drawings from each other.

# With support from![Vlaio](images/vlaio.png) 